# Tutorial 1.2: Experiment Tracking for LLMs
- [source](https://github.com/dmatrix/mlflow-genai-tutorials/blob/main/02_experiment_tracking.ipynb)

![](images/3_Notebook-12-LLM-Experiment-Tracking.png)


## Tracking GenAI Experiments with MLflow

With your environment set up, this notebook covers how to track LLM experiments systematically.

### What You'll Learn
- How `mlflow.openai.autolog()` captures model params, tokens, latency, and I/O automatically
- When you still need explicit `mlflow.log_*` calls (tags, custom artifacts)
- How to create and organize GenAI experiments
- How to compare different LLM configurations
- Practices for keeping experiments organized

### Prerequisites
- Completed Notebook 1.1 (Setup)
- MLflow >= 3.10.0
- MLflow UI running (recommended)

### Estimated Time: 25-30 minutes

---
## Step 1: Environment Setup

Let's load our environment and enable autologging for OpenAI.

In [0]:
%pip install litellm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 44.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 132.4 MB/s  0:00:00
  Attempting uninstall: botocore
    Found existing installation: botocore 1.40.46
    Not uninstalling botocore at /opt/databricks-environments/databricks-ml/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-04848cee-00fe-4d5d-850c-b99984ec666a
    Can't uninstall 'botocore'. No files were found to uninstall.
  Attempting uninstall: aiohttp
    Found existing installation: aiohttp 3.13.2
    Not uninstalling aiohttp at /opt/databricks-environments/databricks-ml/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-04848cee-00fe-4d5d-850c-b99984ec666a
    Can't uninstall 'aiohttp'. No files were found to uninstall.
  Attempting uninstall: s3transfer
    Found existing installation: 

In [0]:
import os

import mlflow
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri("databricks")

use_ai_gateway = os.getenv("USE_DATABRICKS_CLIENT") == "True"

# Verify which client to use
if use_ai_gateway:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    client = w.serving_endpoints.get_open_ai_client()
    model_name = "jsd-gpt-5-2"
else:
    client = OpenAI()
    model_name = "gpt-5.2"

# Verify OpenAI key
if not use_ai_gateway and not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

# Enable autologging for OpenAI.
# This automatically creates MLflow Traces that capture:
#   - model, temperature, max_tokens (all API params as span attributes)
#   - full input messages and response content (span I/O)
#   - token counts: prompt_tokens, completion_tokens, total_tokens
#   - latency (via span start/end timestamps)
mlflow.openai.autolog()

print("✅ Environment configured successfully")
print(f"   MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"   Using model: {model_name}")
print("   Autolog: ENABLED")

✅ Environment configured successfully
   MLflow Tracking URI: databricks
   Using model: gpt-5.2
   Autolog: ENABLED


---
## Step 2: Understanding Experiment Tracking

### What is Experiment Tracking?

Experiment tracking captures the inputs, outputs, and context of your LLM experiments:

```
┌──────────────────────────────────────────────────┐
│              EXPERIMENT                          │
│  Name: "sentiment-analysis"                      │
├──────────────────────────────────────────────────┤
│                                                  │
│  RUN 1: gpt-5.2, temp=1.0                        │
│  ├─ Parameters: {model, temperature, ...}        │
│  ├─ Metrics: {accuracy, latency, cost}           │
│  └─ Artifacts: {prompt.txt, config.json}         │
│                                                  │
│  RUN 2: gpt-5.2, temp=1.5                        │
│  ├─ Parameters: {model, temperature, ...}        │
│  ├─ Metrics: {accuracy, latency, cost}           │
│  └─ Artifacts: {prompt.txt, config.json}         │
│                                                  │
│  RUN 3: gpt-5.2, temp=2.0                        │
│  ...                                             │
└──────────────────────────────────────────────────┘
```

### Key Concepts

- **Parameters**: Configuration values (model name, temperature, max_tokens)
- **Metrics**: Numerical measurements (accuracy, latency, token count)
- **Artifacts**: Files (prompts, responses, model configs)
- **Tags**: Metadata for organizing and filtering runs (e.g. mlflow log tags for system tracing)
- **Traces**: Automatic records of LLM calls (created by autolog)

### What Does `mlflow.openai.autolog()` Capture?
- Note: If we were using a different library or API we would use that instead.
- As an example if using langgraph it wouldbe: `mlflow.langgraph.autolog()`, etc...

Since we enabled autolog in Step 1, every OpenAI call is automatically traced. Here is what you get for free vs. what still needs manual logging:

| Captured Automatically (Traces)             | Requires Explicit `log_*` Calls          |
|---------------------------------------------|------------------------------------------|
| model, temperature, max_tokens (span attrs) | Estimated cost in USD                    |
| Full input messages (span I/O)              | Semantic tags (task, stage, team, etc.)  |
| Full response content (span I/O)            | Structured config artifacts (`log_dict`) |
| Token counts: prompt, completion, total     | Custom business metrics                  |
| Latency (span start/end timestamps)         |                                          |

**The rest of this notebook demonstrates each category — you'll see when a `log_*` call earns its keep.**

---
## Step 3: Your First Tracked LLM Call — The Autolog Way

Let's see what autolog captures with zero manual instrumentation, then add only what it cannot provide.

In [0]:
# Create an experiment -- this is directly in databricks mlflow
mlflow.set_tracking_uri("databricks")
experiment_name = "/Users/adam.m.lang@gmail.com/02-basic-llm-calls"
mlflow.set_experiment(experiment_name)

print(f"📊 Experiment: {experiment_name}")

📊 Experiment: /Users/adam.m.lang@gmail.com/02-basic-llm-calls


### Autolog does the heavy lifting vs. specific params to log
- If you don't use `autolog()` then you would have to set which specific params below you want to log. 

In [0]:
# Make a tracked LLM call — autolog does the heavy lifting.
#
# What we do NOT need to log manually (autolog captures all of this):
#   - mlflow.log_param("model", ...)        -> span attribute
#   - mlflow.log_param("temperature", ...)   -> span attribute
#   - mlflow.log_param("max_tokens", ...)    -> span attribute
#   - mlflow.log_metric("latency_seconds")   -> span timestamps
#   - mlflow.log_metric("prompt_tokens")     -> mlflow.chat.tokenUsage
#   - mlflow.log_metric("completion_tokens") -> mlflow.chat.tokenUsage
#   - mlflow.log_metric("total_tokens")      -> mlflow.chat.tokenUsage
#   - mlflow.log_text(prompt, "prompt.txt")  -> span input
#   - mlflow.log_text(answer, "response.txt")-> span output

## 1. Prompt to test
prompt = "Explain MLflow GenAI Platform in 3-4 sentences."

## 2. Start mlflow run
with mlflow.start_run(run_name="first-llm-tracked-call") as run:

    # The only explicit call: a semantic tag that autolog cannot infer.
    mlflow.set_tag("task", "explanation")

    ## model response
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
        max_completion_tokens=1000
    )
    ## answer
    answer = response.choices[0].message.content

print(f"\n📝 Prompt: {prompt}")
print(f"\n🤖 Response: {answer}")
print(f"\n🔗 Run ID: {run.info.run_id}")
print(f"   View in UI: http://localhost:5000/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")


📝 Prompt: Explain MLflow GenAI Platform in 3-4 sentences.

🤖 Response: MLflow GenAI Platform is a set of MLflow capabilities for building, evaluating, and deploying generative AI (LLM) applications in a consistent, production-oriented workflow. It helps you track prompts, models, and parameters; run standardized evaluations (including LLM-based judges and metrics); and compare results across experiments. It also supports packaging and serving GenAI apps (e.g., prompt chains or retrieval-augmented generation pipelines) with reproducible artifacts and governance. Overall, it brings experiment tracking, evaluation, and deployment discipline to GenAI development.

🔗 Run ID: e2fbb1a7bce9497490264d1841543a24
   View in UI: http://localhost:5000/#/experiments/3442376640170402/runs/e2fbb1a7bce9497490264d1841543a24


Trace(trace_id=tr-3305384c4e325be46a6e750806311dec)

### What Just Happened?

1. **One tag.** We call `set_tag("task", ...)` because that's a semantic label *we* know — autolog has no way to infer it.

2. **Autolog created a Trace automatically.** In the MLflow UI, go to the **Traces** tab of the `02-basic-llm-calls` experiment. You'll see:
   - `model`, `temperature`, `max_tokens` as span attributes
   - The full prompt and response captured as span I/O
   - Token counts under `mlflow.chat.tokenUsage`
   - Token costs for input and output
   - Latency derived from span start/end timestamps

3. **The Trace is linked to the Run** via `mlflow.sourceRun`. You can navigate from the Trace back to the Run and vice versa.

**Try it:** Open the MLflow UI, find the run, and click into its Trace. Compare what autolog captured against what we logged manually.

- The mlflow UI shows us this as part of the trace so we can track metrics on a granular level:

```
completion_tokens: 127
prompt_tokens: 19
total_tokens: 146
completion_tokens_details:
  accepted_prediction_tokens: 0
  audio_tokens: 0
  reasoning_tokens: 0
  rejected_prediction_tokens: 0
prompt_tokens_details:
  audio_tokens: 0
  cached_tokens: 0


```

---
## Step 4: Comparing Multiple Configurations

With autolog active, our comparison helper needs only to make the API call. 

In [0]:
# Simplified helper — autolog captures params, tokens, latency, and I/O automatically.
## helper function allows us to call particular LLM with set temperature
def simple_llm_call(prompt, model=model_name, temperature=1.0, max_completion_tokens=1000, run_name=None):
    """
    Make an LLM call inside a nested run.
    autolog captures model params, token counts, latency, and full I/O as a Trace.
    The run exists only to group and name the experiment.
    """
    with mlflow.start_run(run_name=run_name, nested=True):
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_completion_tokens=max_completion_tokens
        )
        return response.choices[0].message.content

print("✅ Helper function defined!")

✅ Helper function defined!


In [0]:
# Create a new experiment for comparison
mlflow.set_experiment("/Users/adam.m.lang@gmail.com/02-temperature-comparison")

## 1. New Test Prompt
test_prompt = "Write a creative tagline for an AI observability with MLflow GenAI platform."

## 2. Three different test temp configurations -- increasing probabilisticity
temperatures = [1.0, 1.5, 2.0]

print("🔬 Running temperature comparison...\n")

## 3. A parent run groups all nested calls together in the UI.
with mlflow.start_run(run_name="temperature-sweep"):
    mlflow.set_tag("sweep_variable", "temperature")

    for temp in temperatures:
        print(f"  temperature={temp} ...")
        response = simple_llm_call(
            prompt=test_prompt,
            model=model_name,
            temperature=temp,
            max_completion_tokens=1000,
            run_name=f"temp_{temp}"
        )
        print(f"    -> {response}\n")

print("✅ Done. Compare traces side-by-side in the MLflow UI.")

2026/09/10 15:51:39 INFO mlflow.tracking.fluent: Experiment with name '/Users/adam.m.lang@gmail.com/02-temperature-comparison' does not exist. Creating a new experiment.


🔬 Running temperature comparison...

  temperature=1.0 ...
    -> “See every prompt, trace every token—MLflow GenAI Observability keeps your AI reliable from experiment to production.”

  temperature=1.5 ...
    -> “See every prompt, trace every turn—MLflow GenAI Observability keeps your AI trustworthy from experiment to production.”

  temperature=2.0 ...
    -> **“See every prompt, trace every token—ship GenAI you can trust with MLflow Observability.”**

✅ Done. Compare traces side-by-side in the MLflow UI.


[Trace(trace_id=tr-297030f1ddbb524e95270008b13aa962), Trace(trace_id=tr-dafde5ebfbd685571b31551a8553fce8), Trace(trace_id=tr-7288bc9c26c03b93fbaecb58c983cdd1)]

### Analysis

Notice how temperature affects:
- **Creativity**: higher temperature gives more varied responses
- **Consistency**: lower temperature is more deterministic
- **Token usage**: can vary with the creativity level

**Comparing in the MLflow UI:**
1. Select the "02-temperature-comparison" experiment
2. Select the nested runs and click "Compare"
3. Because autolog captured `temperature` as a span attribute on every Trace, you can also filter by temperature directly in the **Traces** tab

---
## Step 5: Tracking Cost Estimates

- As of MLflow 3.10, traces record how much each trace costs, broken down into input and output cost. 
- **This makes it easy to compare overall spend when trying out different models, whether from the same provider or different ones.**


In [0]:
## helper function to make LLM calls with cost tracking
def llm_call_with_cost(prompt, model=model_name, temperature=1.0, max_completion_tokens=1000, run_name=None):
    """
    Make an LLM call with cost tracking.
    autolog captures: model, temperature, token counts, latency, I/O.
    """
    with mlflow.start_run(run_name=run_name):
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_completion_tokens=max_completion_tokens
        )

        answer = response.choices[0].message.content

        return answer

print("✅ Cost-aware helper defined")
print("   autolog captures: model, temperature, token counts, latency, I/O")

✅ Cost-aware helper defined
   autolog captures: model, temperature, token counts, latency, I/O


### Comparing Costs across different models

In [0]:
# Compare costs across different models
mlflow.set_experiment("/Users/adam.m.lang@gmail.com/03-model-cost-comparison")

## 1. Prompt to summarize benefits of MLflow GenAI Platform
prompt = "Summarize the benefits of experiment tracking in 3 bullet points."

## 2. Models to test -- ai gateway or openai
models_to_test = ["jsd-gpt-5-2", "jsd-gpt-5-mini"] if use_ai_gateway else ["gpt-5-mini", "gpt-5.2"]

print("💰 Comparing costs across models...\n")

for model in models_to_test:
    print(f"Testing {model}...")
    response = llm_call_with_cost(
        prompt=prompt,
        model=model,
        temperature=1.0,
        max_completion_tokens=1000,
        run_name=f"model_{model}_run"
    )
    print(f"  Response: {response}...\n")

print("✅ Cost comparison complete! View in MLflow UI.")

2026/09/10 16:01:27 INFO mlflow.tracking.fluent: Experiment with name '/Users/adam.m.lang@gmail.com/03-model-cost-comparison' does not exist. Creating a new experiment.


💰 Comparing costs across models...

Testing gpt-5-mini...
  Response: - Reproducibility and auditability: records code, data, hyperparameters, metrics, and artifacts so experiments can be exactly reproduced, traced, and validated.
- Faster iteration and collaboration: easy comparison and visualization of runs, shared dashboards, and searchable histories speed debugging and team coordination.
- Better model selection and resource efficiency: objective, repeatable comparisons (and automated tracking of tuning) surface the best models faster and reduce wasted compute....

Testing gpt-5.2...
  Response: - **Reproducibility & auditability:** Automatically records parameters, code versions, data references, and results so you can reliably reproduce runs and trace how a model was produced.  
- **Faster iteration & better decisions:** Enables easy comparison across experiments (metrics, artifacts, configs), helping you identify what works, avoid repeated work, and converge faster.  
- **Collab

[Trace(trace_id=tr-e804cb3b5c0d1b16a73cdb509fcd794d), Trace(trace_id=tr-2e402ccad7baa1ac6dde45fcbfa4f6d6)]

### Cost Analysis

Tracking cost lets you:
1. Budget for production deployments
2. Compare model choices (GPT-5-mini vs GPT-5.2)
3. Identify expensive prompts that need optimization
4. Track spending trends over time

**Note:** Token counts come from autolog Traces (`mlflow.chat.tokenUsage`). The only thing `estimated_cost_usd` adds is the dollar figure, which requires pricing data only you can supply — the general pattern is to log exactly what the observability infrastructure can't derive on its own.

---
## Step 6: Organizing Experiments with Tags and Metadata

Tags and structured configs are another area where explicit logging adds real value — autolog has no way to know your team structure, production candidacy, or versioning scheme.

In [0]:
# Systematic experiment with rich metadata — tags and config artifacts.
mlflow.set_experiment("/Users/adam.m.lang@gmail.com/04-production-candidate-testing")

# Test configurations
open_configs = [
    {
        "name": "baseline",
        "model": "gpt-5-mini",
        "temperature": 1.0,
        "system_prompt": "You are a helpful assistant."
    },
    {
        "name": "creative",
        "model": "gpt-5.2",
        "temperature": 2.0,
        "system_prompt": "You are a creative writing assistant."
    },
]

# Databricks hosted foundational models if you want to test them
databricks_config = [
    {
        "name": "baseline",
        "model": "jsd-gpt-5-mini",
        "temperature": 1.0,
        "system_prompt": "You are a helpful assistant."
    },
    {
        "name": "creative",
        "model": "jsd-gpt-5.2",
        "temperature": 1.5,
        "system_prompt": "You are a creative writing assistant."
    },
]
model_configs = databricks_config if use_ai_gateway else open_configs
test_prompt = "Explain the concept of LLM temperature."

print("🏷️  Running experiments with semantic tags...\n")

for config in model_configs:
    with mlflow.start_run(run_name=config["name"]):

        # Make the call — autolog captures model, temperature, tokens, I/O, latency.
        response = client.chat.completions.create(
            model=config["model"],
            messages=[
                {"role": "system", "content": config["system_prompt"]},
                {"role": "user", "content": test_prompt}
            ],
            temperature=config["temperature"],
            max_completion_tokens=1000
        )

        # Log only what autolog cannot provide: semantic tags and structured config.
        mlflow.set_tags({
            "config_name": config["name"],
            "task": "explanation",
            "stage": "testing",
            "team": "ai-research",
            "version": "v1.0",
            "production_candidate": str(config["name"] == "baseline").lower(),
        })

        # Save full config as a structured artifact
        mlflow.log_dict(config, "config.json")

        print(f"  ✓ {config['name']} done")

print("\n✅ All runs completed! Filter by tag 'production_candidate=true' in the UI.")

2026/09/10 16:17:23 INFO mlflow.tracking.fluent: Experiment with name '/Users/adam.m.lang@gmail.com/04-production-candidate-testing' does not exist. Creating a new experiment.


🏷️  Running experiments with semantic tags...

  ✓ baseline done
  ✓ creative done

✅ All runs completed! Filter by tag 'production_candidate=true' in the UI.


[Trace(trace_id=tr-b16445841cbd6426dbad55ac9be12d99), Trace(trace_id=tr-5210fc7327a3f36281c2fefe4c787fe0)]

### Tagging Best Practices

Use tags for:
1. **Environment**: `stage: development/testing/production`
2. **Ownership**: `team: ai-research`, `owner: jules`
3. **Purpose**: `task: summarization`, `use_case: customer-support`
4. **Status**: `production_candidate: true`, `approved: false`
5. **Version**: `version: v1.0`, `prompt_version: v2.1`
6. **Don't duplicate autolog data.** Tags like `model_used: gpt-5.2` or `total_tokens: 342` are already in the Trace. Reserve tags for information that isn't derivable from the API call itself.

You can filter and search runs by tags in the MLflow UI.

---
## Step 7: Querying Experiments Programmatically

Let's learn how to retrieve and analyze experiment data using the MLflow API.

In [0]:
from mlflow.tracking import MlflowClient

# Use the MlflowClient to query experiments and runs
mlflow_client = MlflowClient()

# Get experiment by name
experiment = mlflow_client.get_experiment_by_name("04-production-candidate-testing")

if experiment:
    print(f"📊 Experiment: {experiment.name}")
    print(f"   ID: {experiment.experiment_id}")

    # Search runs — sort by start_time since autolog stores latency on Traces, not run metrics.
    runs = mlflow_client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
        max_results=5
    )

    print(f"\n   Found {len(runs)} runs:\n" + "="*60)

    for run in runs:
        print(f"\n   Run: {run.info.run_name}")
        if run.data.params:
            print("   Parameters:")
            for key, value in run.data.params.items():
                print(f"      {key}: {value}")
        if run.data.metrics:
            print("   Metrics:")
            for key, value in run.data.metrics.items():
                print(f"      {key}: {value}")
        if run.data.tags.get("config_name"):
            print(f"   Tag config_name: {run.data.tags['config_name']}")
else:
    print("Experiment not found. Make sure you ran the production candidate testing section.")

Experiment not found. Make sure you ran the production candidate testing section.


In [0]:
# Find production candidates using tag filters
if experiment:
    prod_runs = mlflow_client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.production_candidate = 'true'",
        max_results=5
    )

    print("🏆 Production Candidates:")
    for run in prod_runs:
        print(f"   Name: {run.info.run_name}")
        print(f"   Config: {run.data.tags.get('config_name', 'N/A')}")
        print(f"   Run ID: {run.info.run_id}")

### Advanced Queries
- Since autolog stores model params, token counts, and latency on Traces rather than run metrics, use mlflow.search_traces() when you need to query that data.

In [0]:
# Get the experiment ID
experiment = mlflow_client.get_experiment_by_name("/Users/adam.m.lang@gmail.com/04-production-candidate-testing")
experiment_id = experiment.experiment_id

# Filter by metric threshold
# Querying Runs (explicit log_* data lives here):
cheap_runs = mlflow_client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="metrics.estimated_cost_usd < 0.001"
)

# Filter by tag
prod_candidates = mlflow_client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.production_candidate = 'true'"
)

In [0]:
# Search traces for an experiment
# Querying Traces (autolog data lives here):
traces = mlflow.search_traces(
    experiment_ids=[experiment_id],
)

# Get a run_id from the first trace
run_id = traces.iloc[0]["trace_metadata"]["mlflow.sourceRun"]

# Get all traces linked to a specific run
traces = mlflow.search_traces(
    run_id=run_id
)

/home/spark-04848cee-00fe-4d5d-850c-b9/.ipykernel/65/command-7687637381245157-1096199686:3: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(


---
## Summary

In this notebook, you learned:

1. How `mlflow.openai.autolog()` captures model params, tokens, latency, and I/O as Traces
2. When to use explicit `mlflow.log_*` calls (cost, tags, config artifacts, semantic outputs)
3. How to compare multiple model configurations with minimal boilerplate
4. How cost is tracked automatically in MLflow 3.10
5. How to organize experiments with tags and metadata
6. How to query runs and traces programmatically

### What to Log and What to Skip

| Category                        | Autolog? | Explicit `log_*`? |
|---------------------------------|:--------:|:------------------:|
| Model, temperature, max_tokens  | ✅ auto  | No — redundant     |
| Token counts (prompt/completion) | ✅ auto | No — redundant     |
| Latency                         | ✅ auto (span timestamps) | No — redundant |
| Full prompt & response          | ✅ auto (span I/O) | No — redundant |
| Semantic tags (task, stage, team)| ❌      | ✅ YES             |
| Structured config artifacts     | ❌       | ✅ YES             |
| Cross-step summaries            | ❌       | ✅ YES             |
| Semantic output params          | ❌       | ✅ YES             |

### Next Steps

**Notebook 1.3: Introduction to Tracing**
- Automatic tracing with autologging
- The trace data model
- Visualizing LLM execution flows
- Integrating with multiple frameworks